# Test YOLO Deteksi & Counting Kendaraan (Lokal)

Notebook ini menguji model `best.pt` secara lokal:
- Deteksi objek pada gambar dataset (val)
- Hitung distribusi kelas (mobil, bus, truk)
- Visualisasi bounding box
- Simulasi counting kendaraan pada frame video

**Tidak ada koneksi ke internet / Cloudinary / ngrok.**

## Cell 1 — Setup Path

In [ ]:
import os
from pathlib import Path

# Sesuaikan jika folder kamu berbeda
YOLO_DIR   = Path(os.path.abspath(''))          # folder yolo/ ini sendiri
MODEL_PATH = YOLO_DIR / 'models' / 'best.pt'

# Dataset — lokasi di mesin lokal
DATASET_DIR = Path('/home/rifki/Documents/belajar/ptkj/coba/pktj-project/backend/yolo/data')
VAL_IMAGES  = DATASET_DIR / 'images' / 'val'
VAL_LABELS  = DATASET_DIR / 'labels' / 'val'
TRAIN_IMAGES = DATASET_DIR / 'images' / 'train'
TRAIN_LABELS = DATASET_DIR / 'labels' / 'train'

CLASS_NAMES = {0: 'mobil', 1: 'bus', 2: 'truk'}
CLASS_COLORS = {
    'mobil': (0, 255, 0),
    'bus':   (255, 100, 0),
    'truk':  (0, 100, 255),
}

print(f'Model  : {MODEL_PATH} — exists: {MODEL_PATH.exists()}')
print(f'Val    : {VAL_IMAGES} — exists: {VAL_IMAGES.exists()}')
print(f'Train  : {TRAIN_IMAGES} — exists: {TRAIN_IMAGES.exists()}')
val_imgs = list(VAL_IMAGES.glob('*.jpg')) + list(VAL_IMAGES.glob('*.png'))
train_imgs = list(TRAIN_IMAGES.glob('*.jpg')) + list(TRAIN_IMAGES.glob('*.png'))
print(f'Jumlah gambar val  : {len(val_imgs)}')
print(f'Jumlah gambar train: {len(train_imgs)}')

## Cell 2 — Load Model

In [ ]:
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))
print(f'Model loaded: {MODEL_PATH.name}')
print(f'Device      : CPU')
print(f'Classes     : {model.names}')

## Cell 3 — Analisis Label Dataset (Ground Truth)

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

def hitung_label(label_dir):
    counter = Counter()
    for lf in Path(label_dir).glob('*.txt'):
        for line in lf.read_text().strip().splitlines():
            if line:
                cls_id = int(line.split()[0])
                counter[CLASS_NAMES.get(cls_id, str(cls_id))] += 1
    return counter

train_count = hitung_label(TRAIN_LABELS)
val_count   = hitung_label(VAL_LABELS)

print('=== Ground Truth Dataset ===')
print('Train:', dict(train_count))
print('Val  :', dict(val_count))
print(f'Total train: {sum(train_count.values())} objek')
print(f'Total val  : {sum(val_count.values())} objek')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (title, cnt) in zip(axes, [('Train', train_count), ('Val', val_count)]):
    keys = list(CLASS_NAMES.values())
    vals = [cnt.get(k, 0) for k in keys]
    bars = ax.bar(keys, vals, color=['#2ecc71', '#3498db', '#e67e22'])
    ax.set_title(f'Distribusi Kelas - {title}')
    ax.set_ylabel('Jumlah Objek')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(val), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(str(YOLO_DIR / 'test_output_distribusi.png'), dpi=100)
plt.show()
print('Saved: test_output_distribusi.png')

## Cell 4 — Deteksi pada Gambar Val (Semua)

In [ ]:
from collections import defaultdict

CONF = 0.25

pred_counter   = Counter()
gt_counter     = Counter()
per_image_stats = []

for img_path in sorted(val_imgs):
    # Ground truth
    lbl_path = VAL_LABELS / (img_path.stem + '.txt')
    gt = Counter()
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            if line:
                gt[CLASS_NAMES.get(int(line.split()[0]), '?')] += 1

    # Prediksi
    results = model.predict(str(img_path), device='cpu', conf=CONF, verbose=False)
    pred = Counter()
    if results[0].boxes is not None:
        for cls_id in results[0].boxes.cls.cpu().numpy().astype(int):
            pred[CLASS_NAMES.get(cls_id, str(cls_id))] += 1

    pred_counter += pred
    gt_counter   += gt
    per_image_stats.append({'file': img_path.name, 'gt': dict(gt), 'pred': dict(pred)})

print('=== Hasil Deteksi pada Val Set ===')
print(f'Ground Truth : {dict(gt_counter)}')
print(f'Prediksi     : {dict(pred_counter)}')
print()
for cls in CLASS_NAMES.values():
    gt_n   = gt_counter.get(cls, 0)
    pred_n = pred_counter.get(cls, 0)
    ratio  = pred_n / gt_n * 100 if gt_n > 0 else 0
    print(f'{cls:6s} → GT: {gt_n:3d} | Pred: {pred_n:3d} | Recall-like: {ratio:.1f}%')

## Cell 5 — Visualisasi Bounding Box (Sample 6 Gambar)

In [ ]:
import cv2
import numpy as np

SAMPLE_COUNT = 6
sample_imgs  = sorted(val_imgs)[:SAMPLE_COUNT]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, img_path in enumerate(sample_imgs):
    frame = cv2.imread(str(img_path))
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    h, w = frame.shape[:2]

    # Ground truth (warna putih putus-putus)
    lbl_path = VAL_LABELS / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.split()
            cls_id = int(parts[0])
            cx, cy, bw, bh = float(parts[1])*w, float(parts[2])*h, float(parts[3])*w, float(parts[4])*h
            x1, y1 = int(cx - bw/2), int(cy - bh/2)
            x2, y2 = int(cx + bw/2), int(cy + bh/2)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (220, 220, 220), 1)
            cv2.putText(frame, f'GT:{CLASS_NAMES.get(cls_id,"?")}',
                        (x1, y1-4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (220, 220, 220), 1)

    # Prediksi
    results = model.predict(str(img_path), device='cpu', conf=CONF, verbose=False)
    n_detected = 0
    if results[0].boxes is not None:
        boxes  = results[0].boxes.xyxy.cpu().numpy()
        clss   = results[0].boxes.cls.cpu().numpy().astype(int)
        confs  = results[0].boxes.conf.cpu().numpy()
        for box, cls_id, conf in zip(boxes, clss, confs):
            x1, y1, x2, y2 = map(int, box)
            cls_name = CLASS_NAMES.get(cls_id, str(cls_id))
            color = CLASS_COLORS.get(cls_name, (255, 255, 0))
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            label = f'{cls_name} {conf:.2f}'
            cv2.putText(frame, label, (x1, y1-8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            n_detected += 1

    axes[i].imshow(frame)
    axes[i].set_title(f'{img_path.name} | Deteksi: {n_detected}', fontsize=9)
    axes[i].axis('off')

fig.suptitle('Prediksi (warna) vs Ground Truth (abu-abu)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(YOLO_DIR / 'test_output_visualisasi.png'), dpi=100)
plt.show()
print('Saved: test_output_visualisasi.png')

## Cell 6 — Simulasi Counting per Frame (Crossing Line)

Mensimulasikan counting kendaraan seperti yang dilakukan `api_local.py`:
kendaraan dihitung ketika centroid melewati garis horizontal di tengah frame.

In [ ]:
# Urutkan gambar train sebagai 'frame-frame video' simulasi
train_img_list = sorted(list(TRAIN_IMAGES.glob('*.jpg')) + list(TRAIN_IMAGES.glob('*.png')))

# Tracking state
counted_ids    = set()
track_history  = {}   # object_id -> list of cy
next_id        = 0

count_total   = 0
count_kiri    = Counter()
count_kanan   = Counter()

LINE_OFFSET   = 40
MAX_DIST      = 100
MIN_FRAMES    = 2

print(f'Simulasi counting pada {len(train_img_list)} frame...')
print('(Gunakan frame train sebagai urutan frame video)')
print()

for frame_idx, img_path in enumerate(train_img_list):
    frame = cv2.imread(str(img_path))
    if frame is None:
        continue
    h, w = frame.shape[:2]
    LINE_Y = h // 2

    results = model.predict(str(img_path), device='cpu', conf=CONF, verbose=False)
    detections = []
    if results[0].boxes is not None:
        for box, cls_id, conf in zip(
            results[0].boxes.xyxy.cpu().numpy(),
            results[0].boxes.cls.cpu().numpy().astype(int),
            results[0].boxes.conf.cpu().numpy()
        ):
            x1, y1, x2, y2 = box
            cx, cy = int((x1+x2)/2), int((y1+y2)/2)
            detections.append({'cx': cx, 'cy': cy, 'cls': CLASS_NAMES.get(cls_id,'mobil'), 'conf': float(conf)})

    # Simple centroid matching
    matched = set()
    for det in detections:
        best_id, best_dist = None, MAX_DIST
        for tid, hist in track_history.items():
            if tid in matched:
                continue
            last_cx, last_cy = hist['positions'][-1]
            d = np.sqrt((det['cx']-last_cx)**2 + (det['cy']-last_cy)**2)
            if d < best_dist:
                best_dist, best_id = d, tid
        if best_id is not None:
            track_history[best_id]['positions'].append((det['cx'], det['cy']))
            track_history[best_id]['cls'] = det['cls']
            matched.add(best_id)
            tid = best_id
        else:
            tid = next_id
            next_id += 1
            track_history[tid] = {'positions': [(det['cx'], det['cy'])], 'cls': det['cls'], 'counted': False}
            matched.add(tid)

        # Cek crossing
        hist = track_history[tid]
        if not hist['counted'] and len(hist['positions']) >= MIN_FRAMES:
            prev_cy = hist['positions'][-2][1]
            curr_cy = hist['positions'][-1][1]
            if prev_cy <= LINE_Y and curr_cy > LINE_Y:
                hist['counted'] = True
                count_total += 1
                lane = 'kiri' if det['cx'] < w//2 else 'kanan'
                if lane == 'kiri':
                    count_kiri[det['cls']] += 1
                else:
                    count_kanan[det['cls']] += 1

print('=== Hasil Simulasi Counting ===')
print(f'Total kendaraan dihitung : {count_total}')
print()
print('Lajur KIRI:')
for cls in CLASS_NAMES.values():
    print(f'  {cls:6s}: {count_kiri.get(cls, 0)}')
print(f'  Total : {sum(count_kiri.values())}')
print()
print('Lajur KANAN:')
for cls in CLASS_NAMES.values():
    print(f'  {cls:6s}: {count_kanan.get(cls, 0)}')
print(f'  Total : {sum(count_kanan.values())}')

## Cell 7 — Visualisasi Hasil Counting

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Total per kelas
classes = list(CLASS_NAMES.values())
totals  = [count_kiri.get(c,0) + count_kanan.get(c,0) for c in classes]
colors  = ['#2ecc71', '#3498db', '#e67e22']
bars = axes[0].bar(classes, totals, color=colors)
axes[0].set_title('Total Kendaraan per Kelas', fontweight='bold')
axes[0].set_ylabel('Jumlah')
for bar, val in zip(bars, totals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 str(val), ha='center', fontweight='bold')

# Plot 2: Per lajur
x = np.arange(len(classes))
width = 0.35
kiri_vals  = [count_kiri.get(c,0) for c in classes]
kanan_vals = [count_kanan.get(c,0) for c in classes]
axes[1].bar(x - width/2, kiri_vals,  width, label='Kiri',  color='#1abc9c')
axes[1].bar(x + width/2, kanan_vals, width, label='Kanan', color='#9b59b6')
axes[1].set_title('Distribusi per Lajur', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(classes)
axes[1].legend()
axes[1].set_ylabel('Jumlah')

# Plot 3: Pie chart keseluruhan
if sum(totals) > 0:
    axes[2].pie(totals, labels=classes, colors=colors,
                autopct='%1.1f%%', startangle=90)
    axes[2].set_title('Proporsi Kelas', fontweight='bold')
else:
    axes[2].text(0.5, 0.5, 'Tidak ada\nkendaraan\nterdeteksi',
                 ha='center', va='center', transform=axes[2].transAxes)
    axes[2].set_title('Proporsi Kelas', fontweight='bold')

fig.suptitle(f'Hasil Simulasi Counting — Total: {count_total} kendaraan',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(YOLO_DIR / 'test_output_counting.png'), dpi=100)
plt.show()
print('Saved: test_output_counting.png')

## Cell 8 — Evaluasi Model (mAP via Ultralytics)

In [ ]:
import yaml

# Tulis data.yaml sementara dengan path absolut
data_yaml_content = {
    'path': str(DATASET_DIR),
    'train': 'images/train',
    'val':   'images/val',
    'names': {0: 'mobil', 1: 'bus', 2: 'truk'}
}
tmp_yaml = YOLO_DIR / 'data_local.yaml'
with open(tmp_yaml, 'w') as f:
    yaml.dump(data_yaml_content, f)

print('Menjalankan evaluasi model (val set)...')
print('(Ini membutuhkan beberapa menit di CPU)\n')

metrics = model.val(
    data=str(tmp_yaml),
    device='cpu',
    conf=0.25,
    iou=0.5,
    verbose=True,
    plots=False,
    save=False,
)

print()
print('=== Metrik Evaluasi ===')
try:
    mp  = metrics.box.mp
    mr  = metrics.box.mr
    map50   = metrics.box.map50
    map5095 = metrics.box.map
    print(f'Precision (mean)  : {mp:.4f}')
    print(f'Recall (mean)     : {mr:.4f}')
    print(f'mAP@0.5           : {map50:.4f}')
    print(f'mAP@0.5:0.95      : {map5095:.4f}')
    print()
    if map50 >= 0.7:
        print('Model BAGUS (mAP@0.5 >= 0.7)')
    elif map50 >= 0.5:
        print('Model CUKUP (mAP@0.5 >= 0.5) — bisa di-improve dengan lebih banyak data')
    else:
        print('Model KURANG (mAP@0.5 < 0.5) — perlu training ulang')
except Exception as e:
    print(f'Tidak bisa baca metrik: {e}')

## Cell 9 — Ringkasan Akhir

In [ ]:
print('=' * 50)
print('  RINGKASAN TEST YOLO LOKAL')
print('=' * 50)
print(f'Model      : {MODEL_PATH.name}')
print(f'Val images : {len(val_imgs)}')
print(f'Train imgs : {len(train_imgs)}')
print()
print('--- Ground Truth Val ---')
for c in CLASS_NAMES.values():
    print(f'  {c:6s}: {val_count.get(c, 0)}')
print()
print('--- Prediksi Val ---')
for c in CLASS_NAMES.values():
    print(f'  {c:6s}: {pred_counter.get(c, 0)}')
print()
print('--- Simulasi Counting (Train frames) ---')
print(f'  Total    : {count_total}')
for c in CLASS_NAMES.values():
    tot = count_kiri.get(c,0) + count_kanan.get(c,0)
    print(f'  {c:6s}   : {tot} (kiri:{count_kiri.get(c,0)}, kanan:{count_kanan.get(c,0)})')
print()
print('--- File Output ---')
for fname in ['test_output_distribusi.png', 'test_output_visualisasi.png', 'test_output_counting.png']:
    fpath = YOLO_DIR / fname
    status = 'OK' if fpath.exists() else 'NOT FOUND'
    print(f'  {fname}: {status}')
print('=' * 50)